# Complete-official-train fine-tuning

This notebook consumes one selected LR, reloads the original pretrained model,
trains on every official training image for 25 epochs, and then invokes the
common repository evaluator once on complete official validation.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


## Configuration


In [ ]:
MODEL_ID = "rtdetrv2_l"
SELECTED_CONFIG = "configs/lr_search/rtdetrv2_l_2class_selected.yaml"

FINAL_EPOCHS = 25
FINAL_SEED = 42
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
ALLOW_OVER_BUDGET_RUN = False
START_EXPENSIVE_STAGE = False
if SMOKE_TEST:
    START_EXPENSIVE_STAGE = False
assert FINAL_EPOCHS == 25 and FINAL_SEED == 42
assert PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS == 8


## Load and validate the selected configuration


In [ ]:
import json
from src.training.lr_workflow import LRControlledBenchmark
from src.utils.serialization import read_yaml

workflow = LRControlledBenchmark(REPO_DIR, DRIVE_ROOT)
workflow.prepare_manifests()
selected_path = REPO_DIR / SELECTED_CONFIG
if not selected_path.exists():
    if SMOKE_TEST:
        print("No selected YAML in smoke mode; run notebook 12 on a GPU first.")
        selected = None
    else:
        raise FileNotFoundError(
            f"{selected_path} is missing. Complete notebook 12 first."
        )
else:
    selected = read_yaml(selected_path)
    assert selected["experiment"]["model_id"] == MODEL_ID
    assert selected["final_training"]["dataset"] == "complete_official_train"
    assert selected["final_training"]["restart_from_pretrained"] is True
    print(selected)


## Prove complete-train identity and validation exclusion


In [ ]:
from src.training.lr_search import assert_final_training_uses_official_train
assert_final_training_uses_official_train(workflow.manifest_dir)
summary = json.loads((workflow.manifest_dir / "split_summary.json").read_text())
print("Complete official train proof:", summary["sources"]["official_train"])
print("Split checks:", summary["verification"])


## Restart from pretrained and run final benchmark


In [ ]:
if START_EXPENSIVE_STAGE:
    if selected is None:
        raise RuntimeError("A selected LR configuration is required.")
    from src.notebook_utils import require_gpu, require_model_environment
    require_model_environment(
        "rtdetr" if MODEL_ID == "rtdetrv2_l" else "openmmlab"
    )
    require_gpu(MODEL_ID)
    final_manifest = workflow.run_final_training(
        MODEL_ID,
        selected_path,
        batch_size=PER_DEVICE_BATCH_SIZE,
        accumulation=GRADIENT_ACCUMULATION_STEPS,
        allow_over_budget_run=ALLOW_OVER_BUDGET_RUN,
        run_common_evaluation=True,
    )
    print("Final registered run:", final_manifest["run_id"])
    print("Output directory:", final_manifest["run_dir"])
    print("Evaluation:", final_manifest.get("final_evaluation_metrics"))
else:
    print("Expensive stage is OFF; no model weights were changed.")
